# Phase 2 Analysis: Sudan Humanitarian Impact Dataset

This notebook loads and summarizes the Phase 2 CIVID dataset for Sudan (initial batch). It mirrors the Phase 1 analysis structure for consistency.

In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import csv
import warnings

warnings.filterwarnings('ignore')

# Helper function to load CSVs robustly
def load_data(file_path):
    return pd.read_csv(
        file_path,
        sep=',',
        quotechar='"',
        quoting=csv.QUOTE_MINIMAL,
        on_bad_lines='warn', # Set to 'warn' to see exactly which lines fail
        low_memory=False,
        encoding='utf-8'
    )

# Load the datasets using the helper function
events_df = load_data('../data/phase2_sudan/events.csv')
persons_df = load_data('../data/phase2_sudan/persons.csv')
sources_df = load_data('../data/phase2_sudan/sources.csv')

print(f"Loaded {len(events_df)} events")
print(f"Loaded {len(persons_df)} person records")
print(f"Loaded {len(sources_df)} sources")

Loaded 11 events
Loaded 3 person records
Loaded 6 sources


## Dataset Overview

In [5]:
print('='*80)
print('PHASE 2 DATASET SUMMARY')
print('='*80)
print(f"\nTotal Events: {len(events_df)}")
print(f"Total Person Records: {len(persons_df)}")
print(f"Sources: {len(sources_df)}")

# Totals where available
total_fatalities = events_df['fatalities'].apply(lambda x: pd.to_numeric(x, errors='coerce')).sum()
total_injuries = events_df['injuries'].apply(lambda x: pd.to_numeric(x, errors='coerce')).sum()

print(f"Total Fatalities: {int(total_fatalities)}")
print(f"Total Injuries: {int(total_injuries)}")

PHASE 2 DATASET SUMMARY

Total Events: 11
Total Person Records: 3
Sources: 6
Total Fatalities: 0
Total Injuries: 0


## Event Distribution and Verification

In [7]:
print('\nEVENTS BY LOCATION TYPE:\n' + '-'*60)
print(events_df['location_type'].value_counts())

print('\nVERIFICATION STATUS BREAKDOWN:\n' + '-'*60)
print(events_df['verification_status'].value_counts())

print('\nCONFIDENCE LEVEL BREAKDOWN:\n' + '-'*60)
print(events_df['confidence_level'].value_counts())


EVENTS BY LOCATION TYPE:
------------------------------------------------------------
location_type
area               5
infrastructure     2
health facility    1
camp               1
territory          1
market area        1
Name: count, dtype: int64

VERIFICATION STATUS BREAKDOWN:
------------------------------------------------------------
verification_status
 verified     6
estimated     3
 estimated    1
verified      1
Name: count, dtype: int64

CONFIDENCE LEVEL BREAKDOWN:
------------------------------------------------------------
confidence_level
high      8
medium    3
Name: count, dtype: int64


## Timeline of Events

In [10]:
events_df['event_date'] = pd.to_datetime(events_df['event_date'], errors='coerce')
events_timeline = events_df.sort_values('event_date')

print('DATE RANGE:', events_timeline['event_date'].min(), 'to', events_timeline['event_date'].max())
print("\nSAMPLE EVENTS (earliest 10):")

for idx, row in events_timeline.head(10).iterrows():
    date_val = row['event_date'].date() if not pd.isna(row['event_date']) else 'unknown'
    print(f"{date_val} | {row['event_id']} | {row['location_type']} | {str(row['location'])[:60]}")

DATE RANGE: 2025-12-31 00:00:00 to 2026-07-08 00:00:00

SAMPLE EVENTS (earliest 10):
2025-12-31 | EVT-030 | infrastructure | Khartoum
2026-05-08 | EVT-024 | infrastructure | Port Sudan to Khartoum humanitarian corridor
2026-06-01 | EVT-020 | area | Al Obeid and North Kordofan
2026-06-01 | EVT-025 | area | Al Obeid and North Kordofan
2026-06-10 | EVT-027 | territory | Khartoum
2026-06-15 | EVT-022 | health facility | Al Obeid Teaching Hospital
2026-06-15 | EVT-026 | area | Tawila town in North Darfur
2026-06-18 | EVT-029 | market area | Gedaref
2026-06-20 | EVT-023 | camp | Al Obeid displacement sites
2026-07-02 | EVT-021 | area | Al Obeid and displacement sites in North Kordofan


## Fatalities by Category (where provided)

In [12]:
# Create a copy to avoid SettingWithCopyWarning
events_with_fatalities = events_df[events_df['fatalities'].notna() & (events_df['fatalities'] != '')].copy()

# Convert to numeric
events_with_fatalities['fatalities'] = events_with_fatalities['fatalities'].apply(lambda x: pd.to_numeric(x, errors='coerce'))

# Group and aggregate
fatalities_by_type = events_with_fatalities.groupby('location_type')['fatalities'].agg(['sum', 'count']).sort_values('sum', ascending=False)

# Print output
print(fatalities_by_type)
print('\nTOTAL FATALITIES (documented where available):', fatalities_by_type['sum'].sum())

Empty DataFrame
Columns: [sum, count]
Index: []

TOTAL FATALITIES (documented where available): 0.0


## Source Reliability

In [13]:
print('\nSOURCES AND RELIABILITY:\n' + '-'*80)
for idx, row in sources_df.iterrows():
    print(f"{row['source_id']:25} | {row['source_type']:15} | {row['reliability_score']:.2f}")


SOURCES AND RELIABILITY:
--------------------------------------------------------------------------------
icrc_sudan                | humanitarian agency | 0.94
who_sudan_health          | health agency   | 0.93
unhcr_sudan_ref           | refugee agency  | 0.92
unocha_sudan_hum          | humanitarian coordinator | 0.91
acled_sudan_conf          | conflict event database | 0.89
batch_note_icrc_acled     | internal_note   | 0.50


## Events Table

The table below lists the Phase 2 events included in this initial batch.

In [14]:
display_cols = ['event_id','event_date','location','location_type','verification_status','confidence_level']
events_display = events_df[display_cols].copy()
events_display['event_date'] = events_display['event_date'].dt.strftime('%Y-%m-%d')
events_display = events_display.sort_values('event_date')
import IPython
IPython.display.display(events_display)

,event_id,event_date,location,location_type,verification_status,confidence_level
9,EVT-030,2025-12-31,Khartoum,infrastructure,verified,high
4,EVT-024,2026-05-08,Port Sudan to Khartoum humanitarian corridor,infrastructure,verified,high
0,EVT-020,2026-06-01,Al Obeid and North Kordofan,area,estimated,high
5,EVT-025,2026-06-01,Al Obeid and North Kordofan,area,verified,high
7,EVT-027,2026-06-10,Khartoum,territory,estimated,medium
2,EVT-022,2026-06-15,Al Obeid Teaching Hospital,health facility,verified,high
6,EVT-026,2026-06-15,Tawila town in North Darfur,area,verified,high
8,EVT-029,2026-06-18,Gedaref,market area,estimated,medium
3,EVT-023,2026-06-20,Al Obeid displacement sites,camp,verified,high
1,EVT-021,2026-07-02,Al Obeid and displacement sites in North Kordofan,area,verified,high


## Notes
- This batch relied primarily on ICRC and ACLED sources after ReliefWeb/WHO/UNHCR pages were inaccessible to automated fetch.
- Entries marked `estimated` should be revisited once primary source reports are accessible.
- Person-level records are included only when individual-level details are provided by sources.